# Small Transformer for Arithmetic

See README.md for setup, results, and limitations.

In [ ]:
import numpy as np
import torch
import random
import sentencepiece as spm  
import os
import matplotlib.pyplot as plt
from pathlib import Path
import itertools
from collections import defaultdict
import json

# Set up environment for CPU usage
os.environ['CUDA_VISIBLE_DEVICES'] = ''

from model import Transformer, ModelArgs
from tokenizer import BatchProcessor, PretokDataset
from demo_training_config import Config
from math_addition_config import create_math_addition_config

In [ ]:
from dataset_generator import generate_datasets_for_ranges

In [ ]:
# Define the ranges to experiment with
ranges = [10, 30, 100, 300]

# Generate datasets for each range using the external dataset generator
datasets = generate_datasets_for_ranges(ranges, data_dir="data")

## 4. Initialize Math Tokenizer 

For arithmetic tasks, we use a custom tokenizer with minimal vocabulary:
- Digits: 0-9 (10 tokens)
- Operators: +, -, = (3 tokens)  
- Space for formatting (1 token)
- Special tokens: <pad>, <bos>, <eos>, <unk> (4 tokens)

Total vocabulary size: **18 tokens** - extremely efficient!

While the tokenizer supports subtraction, we have created separate subtraction datasets for later training. For now, we'll focus exclusively on **addition** training.

In [ ]:
# Import and create our simple math tokenizer
from simple_math_tokenizer import SimpleMathTokenizer

# Create the tokenizer
math_tokenizer = SimpleMathTokenizer()

# Get vocabulary size for model configuration
vocab_size = math_tokenizer.get_vocab_size()
print(f"Math tokenizer initialized with {vocab_size} tokens")

# Display the vocabulary
math_tokenizer.print_vocab()
print()

# Test on various addition expressions (focusing on addition for training)
test_expressions = [
    "1 + 2 = 3",
    "12 + 34 = 46", 
    "123 + 456 = 579",
    "0 + 0 = 0",
    "99 + 1 = 100",
    "300 + 200 = 500"
]

print("Tokenization Examples (Addition Focus):")
print("=" * 50)
for expr in test_expressions:
    math_tokenizer.analyze_expression(expr)

print(f"Vocabulary size: {vocab_size} tokens")
print(f"Current focus: Addition operations (+ and =)")
print(f"Note: Subtraction datasets created separately for future training")

In [ ]:
def parse_val_line(line: str):
    # Expect "a + b = c"
    line = line.strip()
    if not line:
        return None, None
    if "=" in line:
        left, right = line.split("=", 1)
        prompt = left.strip() + " ="
        expected = right.strip()
        return prompt, expected
    return line, ""


val_path = 'data/val_range_30.txt'
with open(val_path, "r", encoding="utf-8") as f:
     # only print first 10 lines
    f = f.readlines()[:10]
    for raw in f:
        prompt, expected = parse_val_line(raw)
        ids = math_tokenizer.encode(prompt + " ", add_bos=True, add_eos=True)
        print(ids, prompt)



In [ ]:
print(math_tokenizer.token_to_id('<eos>'))

## 5. Tokenize and Prepare Training Data

We'll tokenize our datasets using the custom math tokenizer and prepare them for training.

In [ ]:
def tokenize_and_save_dataset(input_file, output_dir, tokenizer, split_name):
    """
    Tokenize dataset and save in binary format expected by training pipeline
    Uses our simple math tokenizer
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Read and tokenize data
    tokens = []
    with open(input_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                # Add BOS and EOS tokens
                token_ids = tokenizer.encode(line, add_bos=True, add_eos=True)
                tokens.extend(token_ids)
    
    # Save as binary file
    tokens_array = np.array(tokens, dtype=np.uint16)
    output_file = os.path.join(output_dir, f"{split_name}.bin")
    tokens_array.tofile(output_file)
    
    print(f"Saved {len(tokens)} tokens to {output_file}")
    return len(tokens)

# Tokenize datasets for each range using our simple math tokenizer
print("Tokenizing datasets with Simple Math Tokenizer...")
print("=" * 50)

for max_val in ranges:
    output_dir = f"pretok_data/simple_math_range_{max_val}"
    print(f"\nProcessing range {max_val}...")
    
    # Tokenize train and val sets
    train_tokens = tokenize_and_save_dataset(
        f'data/train_range_{max_val}.txt', output_dir, math_tokenizer, 'train'
    )
    val_tokens = tokenize_and_save_dataset(
        f'data/val_range_{max_val}.txt', output_dir, math_tokenizer, 'val'
    )
    
    print(f"Range {max_val}: {train_tokens} train tokens, {val_tokens} val tokens")

print(f"\nAll datasets tokenized using Simple Math Tokenizer (vocab_size={vocab_size})")

# Show tokenization statistics
print(f"\nTokenization Statistics:")
print(f"Vocabulary size: {vocab_size}")
print(f"Average tokens per expression:")

sample_expressions = ["1 + 2 = 3", "12 + 34 = 46", "123 + 456 = 579"]
total_length = 0
for expr in sample_expressions:
    tokens = math_tokenizer.encode(expr, add_bos=True, add_eos=True)
    print(f"  '{expr}': {len(tokens)} tokens")
    total_length += len(tokens)

print(f"Average: {total_length / len(sample_expressions):.1f} tokens per expression")